In [109]:
# Matplotlib 中文字体设置（避免中文字符显示警告）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

In [93]:
import numpy as np
from sklearn.model_selection import train_test_split

# 数据导入

In [112]:
#周志华西瓜数据集：根据特征是否好瓜
txt_path=r".\Watermelon.txt"
with open(txt_path, encoding='utf-8') as t:
     result=[]
     lines = t.read().splitlines()
     for i in range(len(lines)):
        line=lines[i].split(',')
        del line[0]
        result.append(line)
array = np.asarray(result)
# 仅保留特征名（去掉最后一列标签名“好瓜”）
data_header = array[0,:-1].tolist()
print('Feature header = ', data_header)
data = array[1:,:-1]
labels = array[1:,-1]
print('Samples =', data.shape[0], 'Features =', data.shape[1])
print('Label examples:', labels[:5])

# 分训练集和测试集（保持标签分布）
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.3, random_state=42, stratify=labels
)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

Feature header =  ['色泽', '根蒂', '敲声', '纹理', '脐部', '触感']
Samples = 17 Features = 6
Label examples: ['是' '是' '是' '是' '是']
Train shape: (11, 6) Test shape: (6, 6)


# 构建决策树

In [95]:
def get_shannon_entropy(labels: np.ndarray) -> float:
    """
    输入：labels：节点中所有数据的标签，用于计算概率
    输出：shannon_entropy：香农熵
    """
    # 统计每个标签出现的次数
    label_counts = {}
    for label in labels:
        if label not in label_counts:
            label_counts[label] = 0
        label_counts[label] += 1
    # 计算香农熵
    shannon_entropy = 0.0
    num = len(labels)
    for count in label_counts.values():
        prob = count / num
        shannon_entropy -= prob * np.log2(prob)
    return shannon_entropy

In [96]:
def get_conditional_entropy(datas: np.ndarray, labels: np.ndarray) -> float:
    """
    输入：datas:具体来说会是某一维特征的数据； labels：所有数据的标签用于计算概率
    输出：conditional_entropy：根据某一维特征划分的条件熵
    """
    # 统计每个特征值对应的样本索引
    value_dict = {}
    for idx, val in enumerate(datas):
        if val not in value_dict:
            value_dict[val] = []
        value_dict[val].append(idx)
    # 计算条件熵
    conditional_entropy = 0.0
    num = len(datas)
    for val, idxs in value_dict.items():
        sub_labels = labels[idxs]
        prob = len(idxs) / num
        conditional_entropy += prob * get_shannon_entropy(sub_labels)
    return conditional_entropy

In [97]:
def get_best_gain(datas: np.ndarray, labels: np.ndarray) -> tuple[int, float]:
    """
    对比根据每一维特征求得的增益Gain，选出最佳特征
    输入：datas：整个数据集；labels：所有数据的标签
    输出：best_feature：当前最佳特征； best_gain：当前最佳增益
    """
    base_entropy = get_shannon_entropy(labels)
    best_gain = -1
    best_feature = -1
    n_features = datas.shape[1]
    for i in range(n_features):
        feature_values = datas[:, i]
        cond_entropy = get_conditional_entropy(feature_values, labels)
        gain = base_entropy - cond_entropy
        if gain > best_gain:
            best_gain = gain
            best_feature = i
    return best_feature, best_gain

In [110]:
from typing import Dict, Any, Union

def create_tree(datas_header: list, datas: np.ndarray, labels: np.ndarray) -> Union[Dict[str, Any], str]:
    """
    递归构建决策树
    输入：datas_header：特征名列表；datas：特征数据；labels：标签
    输出：树的嵌套字典结构（所有分支键和值统一为 Python str，避免 np.str_ 带来的键不匹配）
    返回类型为 Union[Dict[str, Any], str]，因为叶子节点直接返回标签字符串
    另外：在每个节点记录该节点的多数类，用于预测时遇到未见过取值的回退。
    """
    # 当前节点多数类
    values, counts = np.unique(labels, return_counts=True)
    majority_label = str(values[np.argmax(counts)])

    # 结束条件1：所有标签相同
    if len(set(labels)) == 1:
        return str(labels[0])
    # 结束条件2：特征用完，返回出现最多的标签
    if datas.shape[1] == 0:
        return majority_label

    # 选择最佳特征
    best_feature, best_gain = get_best_gain(datas, labels)
    best_feature_name = datas_header[best_feature]

    # 节点结构：{ 特征名: {取值: 子树或叶子, '__majority__': 多数类} }
    tree: Dict[str, Any] = {best_feature_name: {}}
    feature_values = np.unique(datas[:, best_feature])
    for value in feature_values:
        idxs = np.where(datas[:, best_feature] == value)[0]
        sub_datas = np.delete(datas[idxs], best_feature, axis=1)
        sub_labels = labels[idxs]
        sub_header = datas_header[:best_feature] + datas_header[best_feature+1:]
        tree[best_feature_name][str(value)] = create_tree(sub_header, sub_datas, sub_labels)
    # 在分支字典中存入多数类，便于未见过取值时回退
    tree[best_feature_name]['__majority__'] = majority_label
    return tree

In [111]:
def Predict_Results(tree_model, datas, datas_header):
    """
    基于构建的树进行预测
    - 输入值转为 str 与树的键对齐
    - 未见过取值时，优先回退到该节点记录的多数类 '__majority__'，否则回退到第一个合法分支
    """
    def predict_result(trees_model: dict, input_data: list, datas_header: list) -> str:
        # 叶子：直接返回
        if not isinstance(trees_model, dict):
            return str(trees_model)
        # 取当前判断特征
        # 注意：节点结构包含一个特征名键和可能的其它元数据键，但我们构造时先插入特征键
        cur_judge = next(iter(trees_model.keys()))
        cur_tree = trees_model[cur_judge]
        num_feature = datas_header.index(cur_judge)
        cur_val = str(input_data[num_feature])
        # 未见过取值，回退到多数类
        if cur_val not in cur_tree:
            if '__majority__' in cur_tree:
                return str(cur_tree['__majority__'])
            # 否则回退到第一个非元数据分支
            for k in cur_tree.keys():
                if k != '__majority__':
                    cur_val = k
                    break
        next_node = cur_tree[cur_val]
        if not isinstance(next_node, dict):
            return str(next_node)
        return predict_result(next_node, input_data, datas_header)

    results = np.empty(datas.shape[0], dtype=object)
    for i, data in enumerate(datas):
        data = data.tolist() if hasattr(data, "tolist") else list(data)
        results[i] = predict_result(tree_model, data, datas_header)
    return results


def evaluation(y, results):
    num = y.shape[0]
    acc = sum(y == results) / num
    print('Accuracy = ', float(acc))

In [100]:
# 保存和读取函数
def store_tree(input_tree, filename):
    import pickle
    with open(filename, 'wb') as f:
        pickle.dump(input_tree, f)
        f.close()


def restore_tree(filename):
    import pickle
    with open(filename, 'rb') as f:
        return pickle.load(f)


In [113]:
# 创建树
tree_model = create_tree(data_header, X_train, y_train)
print("决策树构建完成")
print(tree_model)

# results = Predict_Results(tree_model, X_test, data_header)
# evaluation(y_test,results)

# # 保存树模型
# store_tree(tree_model, 'Watermelon.pkl')     
# print("模型保存完毕")

决策树构建完成
{'纹理': {'模糊': '否', '清晰': {'根蒂': {'硬挺': '否', '稍蜷': '是', '蜷缩': '是', '__majority__': '是'}}, '稍糊': '否', '__majority__': '否'}}


In [114]:
# 训练集上做一次快速自检
train_pred = Predict_Results(tree_model, X_train, data_header)
evaluation(y_train, train_pred)

Accuracy =  1.0


In [115]:
# 测试集预测与评估（手写ID3）
print("\n[Handmade ID3] Test evaluation")
test_pred = Predict_Results(tree_model, X_test, data_header)
print('Pred:', test_pred)
print('True:', y_test)
evaluation(y_test, test_pred)


[Handmade ID3] Test evaluation
Pred: ['是' '是' '否' '否' '否' '是']
True: ['是' '否' '否' '否' '是' '是']
Accuracy =  0.6666666666666666


In [ ]:
# 信息增益排名
import matplotlib.pyplot as plt

def info_gain_ranking(datas: np.ndarray, labels: np.ndarray, feature_names: list):
    base_entropy = get_shannon_entropy(labels)
    gains = []
    for i in range(datas.shape[1]):
        cond_entropy = get_conditional_entropy(datas[:, i], labels)
        gain = base_entropy - cond_entropy
        gains.append((feature_names[i], float(gain)))
    # 按信息增益从高到低排序
    gains.sort(key=lambda x: x[1], reverse=True)
    return gains

rankings = info_gain_ranking(X_train, y_train, data_header)
print("\n[Info Gain Ranking @ root (train)]")
for idx, (name, gain) in enumerate(rankings, 1):
    print(f"{idx}. {name}: {gain:.4f}")


[Info Gain Ranking @ root (train)]
1. 纹理: 0.6395
2. 脐部: 0.4154
3. 敲声: 0.2577
4. 根蒂: 0.2427
5. 色泽: 0.0759
6. 触感: 0.0013


In [116]:
# 手写ID3：测试集详细对比与可视化（坏瓜/好瓜）
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

label_map = {"是": "好瓜", "否": "坏瓜"}

hand_pred = Predict_Results(tree_model, X_test, data_header)
print("\n[Handmade ID3] classification report (原始标签)")
print(classification_report(y_test, hand_pred, digits=4))

# 表格展示（映射为好瓜/坏瓜便于阅读）
df_test = pd.DataFrame(X_test, columns=data_header)
df_test["真实标签"] = [label_map.get(v, v) for v in y_test]
df_test["预测标签"] = [label_map.get(v, v) for v in hand_pred]
print("\n测试集样本（部分）:")
print(df_test.head(10))


[Handmade ID3] classification report (原始标签)
              precision    recall  f1-score   support

           否     0.6667    0.6667    0.6667         3
           是     0.6667    0.6667    0.6667         3

    accuracy                         0.6667         6
   macro avg     0.6667    0.6667    0.6667         6
weighted avg     0.6667    0.6667    0.6667         6


测试集样本（部分）:
   色泽  根蒂  敲声  纹理  脐部  触感 真实标签 预测标签
0  乌黑  蜷缩  浊响  清晰  凹陷  硬滑   好瓜   好瓜
1  乌黑  稍蜷  浊响  清晰  稍凹  软粘   坏瓜   好瓜
2  浅白  蜷缩  浊响  模糊  平坦  软粘   坏瓜   坏瓜
3  青绿  稍蜷  浊响  稍糊  凹陷  硬滑   坏瓜   坏瓜
4  乌黑  稍蜷  浊响  稍糊  稍凹  软粘   好瓜   坏瓜
5  乌黑  稍蜷  浊响  清晰  稍凹  硬滑   好瓜   好瓜
